In [17]:

import sys
import os
import time
import numpy as np

import MDAnalysis as mda
# from MDAnalysis.analysis import align
# from MDAnalysis.analysis import distances

import westpa
from westpa.analysis import Run

import matplotlib.pyplot as plt

In [18]:
#this barely needs to be a method but having the .h5 stuff compartmentalized is nice
def load_h5_pcs(h5path, miniter, maxiter):
    
    run = Run.open(h5path)

    #set maximum iteration automatically
    if maxiter == -1:
        maxiter = run.num_iterations

    pcs = [iteration.pcoords for iteration in run if (iteration.number >= miniter and iteration.number < maxiter)]

    return pcs

In [45]:
#specify input file

cftr_west = "/home/jonathan/Documents/grabelab/cftr/chloe-data"
cftr_refpc = "/home/jonathan/Documents/grabelab/cftr/refeaturization"

h5paths_names = [[f"{cftr_west}/wstp_cftr_1_degrabo/west-040925.h5", f"{cftr_refpc}/nonlip_glpg_1", "pyrazole-1", "blue"],
                  [f"{cftr_west}/wstp_cftr_2_wynton/west-040925.h5", f"{cftr_refpc}/nonlip_glpg_2", "pyrazole-2", "cyan"],
                  [f"{cftr_west}/wstp_lip_glpg_1/west-040925.h5", f"{cftr_refpc}/lip_glpg_1", "undecanol-1", "red"],
                  [f"{cftr_west}/wstp_lip_glpg_2/west-040925.h5", f"{cftr_refpc}/lip_glpg_2", "undecanol-2", "orange"]]

#westpa rounds to load
minround = 0
maxround = -1

run_ind = 3

data_paths = ["/home/jonathan/Documents/grabelab/cftr/revisions/abbv-974-1",
              "/home/jonathan/Documents/grabelab/cftr/revisions/abbv-974-2",
              "/home/jonathan/Documents/grabelab/cftr/revisions/cftri-c10-1",
              "/home/jonathan/Documents/grabelab/cftr/revisions/cftri-c10-2"]
data_path = data_paths[run_ind]

topology_path_strings = ["nonlip_glpg_1", "nonlip_glpg_2", "lip_glpg_1", "lip_glpg_2"]
topology_path_string = topology_path_strings[run_ind]


In [46]:
pcs_all = load_h5_pcs(h5paths_names[run_ind][0], minround, maxround)

In [47]:
nbins = 51
binbounds = np.arange(0,nbins,1)
prot_wat_contacts_by_bin = [[] for a in range(nbins)]
prot_lip_contacts_by_bin = [[] for a in range(nbins)]
prot_lig_contacts_by_bin = [[] for a in range(nbins)]
lig_prot_contacts_by_bin = [[] for a in range(nbins)]
lig_wat_contacts_by_bin = [[] for a in range(nbins)]
lig_lip_contacts_by_bin = [[] for a in range(nbins)]

#loop over WE rounds
for r in range(1,2000,10):
    
    #get progress coordinates of the walkers, accounting for the occasional corrupted file
    pcs_flat = pcs_all[r-1][:,-1].flatten()
    walkers = np.load(f"{data_path}/pc_data_round_{r}_walker_numbers_v1.npy")
    pcs = [pcs_flat[w] for w in walkers]

    #assign frames to bins by pc
    bins = np.digitize(pcs, binbounds)

    #load water coordinates
    protein_water_contacts = np.load(f"{data_path}/pc_data_round_{r}_prot_wat_contacts_v1.npy")
    protein_lipid_contacts = np.load(f"{data_path}/pc_data_round_{r}_prot_lip_contacts_v1.npy")
    protein_ligand_contacts = np.load(f"{data_path}/pc_data_round_{r}_prot_lig_contacts_v1.npy")
    ligand_water_contacts = np.load(f"{data_path}/pc_data_round_{r}_lig_wat_contacts_v1.npy")
    ligand_lipid_contacts = np.load(f"{data_path}/pc_data_round_{r}_lig_lip_contacts_v1.npy")

    #get coordinates of waters in each bin
    for b, pwc, plc, prot_lig, lwc, llc in zip(bins, protein_water_contacts, protein_lipid_contacts, protein_ligand_contacts, ligand_water_contacts, ligand_lipid_contacts):
        prot_wat_contacts_by_bin[b].append(pwc)
        prot_lip_contacts_by_bin[b].append(plc)
        prot_lig_contacts_by_bin[b].append(np.where(np.sum(prot_lig, axis = 1)>0, 1, 0))
        lig_prot_contacts_by_bin[b].append(np.where(np.sum(prot_lig, axis = 0)>0, 1, 0))
        lig_wat_contacts_by_bin[b].append(lwc)
        lig_lip_contacts_by_bin[b].append(llc)

for i, w in enumerate(prot_wat_contacts_by_bin):
    print(f"{i}: {len(w)}")

prot_wat_contacts_by_bin = [np.mean(np.stack(w), axis=0) if len(w) > 0 else np.array([]) for w in prot_wat_contacts_by_bin]
prot_lip_contacts_by_bin = [np.mean(np.stack(w), axis=0) if len(w) > 0 else np.array([]) for w in prot_lip_contacts_by_bin]
prot_lig_contacts_by_bin = [np.mean(np.stack(w), axis=0) if len(w) > 0 else np.array([]) for w in prot_lig_contacts_by_bin]
lig_prot_contacts_by_bin = [np.mean(np.stack(w), axis=0) if len(w) > 0 else np.array([]) for w in lig_prot_contacts_by_bin]
lig_wat_contacts_by_bin = [np.mean(np.stack(w), axis=0) if len(w) > 0 else np.array([]) for w in lig_wat_contacts_by_bin]
lig_lip_contacts_by_bin = [np.mean(np.stack(w), axis=0) if len(w) > 0 else np.array([]) for w in lig_lip_contacts_by_bin]


0: 0
1: 721
2: 3533
3: 4298
4: 2302
5: 2586
6: 1376
7: 675
8: 393
9: 324
10: 320
11: 366
12: 302
13: 236
14: 188
15: 205
16: 171
17: 203
18: 194
19: 213
20: 169
21: 182
22: 153
23: 137
24: 171
25: 199
26: 269
27: 251
28: 179
29: 147
30: 93
31: 99
32: 77
33: 69
34: 59
35: 75
36: 67
37: 61
38: 58
39: 43
40: 82
41: 77
42: 47
43: 27
44: 13
45: 11
46: 7
47: 4
48: 3
49: 0
50: 0


In [48]:
# print(len(prot_lig_contacts_by_bin[1]))
# print(len(lig_prot_contacts_by_bin[1]))


In [49]:
def tmd_query():
    segment_resis = [[77, 149], [192, 245], [298, 362], [988, 1034], [857, 889], [900, 942], [1094, 1154]]
    #print("color blue, " + " or ".join([f"resi {sr[0]}-{sr[1]}" for sr in segment_resis]))

    segment_resis_all = [i for sr in segment_resis for i in range(sr[0], sr[1]+1)]
    query = " or ".join([f"resid {sr}" for sr in segment_resis_all])

    #indices = frame.top.select(f"protein and ({query})")

    # print_indices = frame.top.select(f"protein and name CA and ({query})")
    # print("+".join([str(i+1) for i in print_indices]))
    return f"protein and ({query})"

In [50]:
#Chatgpt prompt:
"""write a python function that takes as its arguments a path to a pdb file, an mdanalysis atomgroup, and a list of numbers equal in length to the atom group, 
and saves a copy of the pdb file with the b factors of the atoms in the atom group set equal to the numbers in the list. 
Match atoms using residue names and numbers since the indices do not match between the atomgroup and pdb file"""


def write_bfactors_by_residue_match(pdb_path, atomgroup, values, output_pdb=None):
    """
    Write a copy of a PDB file with B-factors set for atoms matching an AtomGroup.
    Matching is done using (resname, resid, atom name), NOT indices.

    Parameters
    ----------
    pdb_path : str
        Path to input PDB file.
    atomgroup : MDAnalysis AtomGroup
        AtomGroup derived from (possibly different) structure.
    values : array-like
        Numbers equal in length to atomgroup.
    output_pdb : str
        Path to output PDB file.
    """

    if output_pdb is None:
        base = os.path.splitext(pdb_path)[0]
        output_pdb = base + "_bfactored.pdb"

    values = np.asarray(values)

    if len(atomgroup) != len(values):
        print("note: this may be caused by not loading data from enough WE rounds so that an empty array is passed as input")
        raise ValueError("Length of values must equal AtomGroup length.")

    # Load fresh universe from PDB to preserve original ordering
    u = mda.Universe(pdb_path)

    # Ensure tempfactors exist
    if not hasattr(u.atoms, "tempfactors"):
        u.add_TopologyAttr("tempfactors")

    # Start from existing B-factors (or zeros)
    try:
        b = u.atoms.tempfactors.copy()
    except AttributeError:
        b = np.zeros(len(u.atoms))

    # Build lookup dictionary from PDB universe
    pdb_lookup = {}
    for atom in u.atoms:
        key = (atom.resname, atom.resid, atom.name)
        pdb_lookup[key] = atom.index

    # Assign B-factors by matching keys
    for atom, value in zip(atomgroup, values):
        key = (atom.resname, atom.resid, atom.name)

        if key not in pdb_lookup:
            raise ValueError(
                f"Atom not found in PDB: resname={atom.resname}, "
                f"resid={atom.resid}, name={atom.name}"
            )

        pdb_index = pdb_lookup[key]
        b[pdb_index] = value

    # Reassign full array (required by MDAnalysis)
    u.atoms.tempfactors = b

    # Write output
    with mda.Writer(output_pdb, multiframe=False) as W:
        W.write(u.atoms)


In [51]:
def ref_prot_seles(toppathstring):
    ref_frame_path = f'/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/{toppathstring}/topology/input.gro'
    u = mda.Universe(ref_frame_path)
    prot_sel = u.select_atoms(f"{tmd_query()} and not name H*")
    lig_sel = u.select_atoms("resname LJP and not name H*")
    return u, prot_sel, lig_sel

u, prot_sel, lig_sel = ref_prot_seles(topology_path_string)

In [52]:
# print(len(prot_sel))
# print(len(pwc))

In [53]:
bins = [1,10,20,30,40]

for bin in bins:
    pwc = prot_wat_contacts_by_bin[bin]
    plc = prot_lip_contacts_by_bin[bin]
    prot_lig = prot_lig_contacts_by_bin[bin]
    lig_prot = lig_prot_contacts_by_bin[bin]
    lwc = lig_wat_contacts_by_bin[bin]
    llc = lig_lip_contacts_by_bin[bin]

    input_file = f"/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/{topology_path_string}/topology/input.pdb"
    #"/home/jonathan/Documents/grabelab/cftr/revisions/abbv-974-1/step5_input_from_gro.pdb"

    if not os.path.exists(f"{data_path}-visualization"):
        os.mkdir(f"{data_path}-visualization")

    write_bfactors_by_residue_match(input_file, prot_sel, pwc,
                                    f"{data_path}-visualization/input_prot_wat_{bin}.pdb")
    write_bfactors_by_residue_match(input_file, prot_sel, plc,
                                    f"{data_path}-visualization/input_prot_lip_{bin}.pdb")

    write_bfactors_by_residue_match(input_file, prot_sel, prot_lig,
                                    f"{data_path}-visualization/input_prot_lig_{bin}.pdb")
    write_bfactors_by_residue_match(input_file, lig_sel, lig_prot,
                                    f"{data_path}-visualization/input_lig_prot_{bin}.pdb")

    write_bfactors_by_residue_match(input_file, lig_sel, lwc,
                                    f"{data_path}-visualization/input_lig_wat_{bin}.pdb")
    write_bfactors_by_residue_match(input_file, lig_sel, llc,
                                    f"{data_path}-visualization/input_lig_lip_{bin}.pdb")
    #view with:
    #hide sticks; show sticks, (poly or resn LJP) and not elem H
    #spectrum b, blue_red

In [ ]:

#                                                     END 

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Example large dataset
# N = 200000
# coords = np.random.randn(N,3)
# x = coords[:,0]
# y = coords[:,1]
# z = coords[:,2]

mean_prot = np.mean(prot_pos, axis = 0)

bin = 5
interval=1

#print(waters_by_bin[bin])

x = waters_by_bin[bin][::interval,0]-mean_prot[0]
y = waters_by_bin[bin][::interval,1]-mean_prot[1]
z = waters_by_bin[bin][::interval,2]-mean_prot[2]

fig = go.Figure()

fig.add_trace(
    go.Scatter3d(
        x=x,
        y=y,
        z=z,
        mode="markers",
        hoverinfo="skip",
        marker=dict(
            size=3,
            opacity=0.7,
            color="red"#z,          # GPU color mapping
            #colorscale="Viridis"
        )
    )
)


if True:
    xp = prot_pos[:,0]-mean_prot[0]
    yp = prot_pos[:,1]-mean_prot[1]
    zp = prot_pos[:,2]-mean_prot[2]

    fig.add_trace(
        go.Scatter3d(
            x=xp,
            y=yp,
            z=zp,
            mode="markers",
            hoverinfo="skip",
            marker=dict(
                size=4,
                opacity=1,
                color="black"
            )
        )
    )

    xl = lig_pos[:,0]-mean_prot[0]
    yl = lig_pos[:,1]-mean_prot[1]
    zl = lig_pos[:,2]-mean_prot[2]

    fig.add_trace(
        go.Scatter3d(
            x=xl,
            y=yl,
            z=zl,
            mode="markers",
            hoverinfo="skip",
            marker=dict(
                size=4,
                opacity=1,
                color="blue"
            )
        )
    )

fig.update_layout(
    title="Large Interactive 3D Scatter",
    scene=dict(
        # xaxis = dict(nticks=4, range=[-30,30],),
        # yaxis = dict(nticks=4, range=[-30,30],),
        # zaxis = dict(nticks=4, range=[-30,30],),
        aspectratio=dict(x=1, y=1, z=1),
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z"
    ),
    width=1000,
    height=1000,
    margin=dict(l=0,r=0,b=0,t=40),
    scene_camera=dict(
            eye=dict(x=3, y=3, z=3)
        )

)

fig.show(config={"scrollZoom": True})